# 01 · Exploratory data analysis
Load the cleaned UCI Online Retail II data, inspect its coverage and create the core business plots.

In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd() if (Path.cwd()/'src').exists() else Path.cwd().parent
sys.path.insert(0,str(ROOT))
from src.data_prep import load_raw, clean_transactions
tx,audit=clean_transactions(load_raw())
audit, tx.shape, tx.InvoiceDate.min(), tx.InvoiceDate.max()

## Revenue trend, countries, products and purchasing behavior
Cancelled invoices and returns are excluded from purchase revenue. Missing customer IDs and exact duplicates are also removed.

In [ ]:
import matplotlib.pyplot as plt
monthly=tx.set_index('InvoiceDate').resample('MS').Revenue.sum()
monthly.plot(title='Monthly revenue'); plt.ylabel('Revenue (£)'); plt.show()
tx.groupby('Country').Revenue.sum().nlargest(10).sort_values().plot.barh(title='Top countries'); plt.show()
tx.groupby('Description').Revenue.sum().nlargest(10).sort_values().plot.barh(title='Top products'); plt.show()

In [ ]:
orders=tx.groupby('Customer ID').Invoice.nunique()
repeat_rate=(orders>=2).mean()
print(f'Repeat-purchase rate: {repeat_rate:.1%}')
orders.value_counts().sort_index().head(25).plot.bar(title='Order frequency per customer'); plt.show()
customer_revenue=tx.groupby('Customer ID').Revenue.sum().sort_values(ascending=False)
cum=customer_revenue.cumsum()/customer_revenue.sum()
print(f'Top 20% customer share: {cum.iloc[max(0,int(.2*len(cum))-1)]:.1%}')
plt.plot(range(1,len(cum)+1),cum); plt.title('Pareto customer revenue curve'); plt.show()

## Business insight checklist
Use the measured tables and plots above to fill five plain-English findings: revenue seasonality; country concentration; product concentration; repeat-purchase distribution; and share of revenue captured by leading customers. Avoid interpreting high sales as profit.